In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import train_test_split

fyp_dir = Path.cwd().parent
file_path = fyp_dir / "data" / "triage.csv" / "triage.csv"

df = pd.read_csv(file_path)

required_columns = [
    "subject_id","stay_id","temperature","heartrate","resprate","o2sat",
    "sbp","dbp","pain","acuity","chiefcomplaint"
]
missing = [c for c in required_columns if c not in df.columns]
print("Missing columns:", missing)

df = df[required_columns].copy()
df.head()
print("Shape before filtering:", df.shape)

Missing columns: []
Shape before filtering: (425087, 11)


In [3]:
df = df[df["acuity"].isin([1,2,3,4,5])].copy()

df["chiefcomplaint"] = df["chiefcomplaint"].astype(str).str.strip()
df = df[df["chiefcomplaint"].notna() & (df["chiefcomplaint"] != "")].copy()

print("Shape after filtering:", df.shape)
df["acuity"].value_counts().sort_index()

Shape after filtering: (418083, 11)


acuity
1.0     24015
2.0    139407
3.0    225060
4.0     28502
5.0      1099
Name: count, dtype: int64

In [4]:
def map_esi_to_risk(esi):
    if esi in [1, 2]:
        return "High"
    elif esi == 3:
        return "Medium"
    elif esi in [4, 5]:
        return "Low"
    return np.nan

df["risk"] = df["acuity"].apply(map_esi_to_risk)

print(df["risk"].value_counts())

risk
Medium    225060
High      163422
Low        29601
Name: count, dtype: int64


4. ** Text length stats (for report) **

In [5]:
df["text_len_words"] = df["chiefcomplaint"].str.split().apply(len)

stats = df.groupby("risk")["text_len_words"].agg(["count","mean","median","min","max"])
stats

,count,mean,median,min,max
risk,,,,,
High,163422,2.351275,2.0,1,16
Low,29601,2.545894,3.0,1,12
Medium,225060,2.496650,2.0,1,14


5. ** Stratified train/test split by risk **


In [ ]:
X = df["chiefcomplaint"]
y = df["risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train class counts:\n", y_train.value_counts())
print("\nTest class counts:\n", y_test.value_counts())

6. ** Save splits  **

In [ ]:
train_df = df.loc[X_train.index].copy()
test_df  = df.loc[X_test.index].copy()

train_df.to_csv(f"{fyp_dir}/data/train_split.csv", index=False)
test_df.to_csv(f"{fyp_dir}/data/test_split.csv", index=False)

print("Saved: train_split.csv and test_split.csv")